### LIBRARY IMPORTS

In [12]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy
import warnings

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.nn_regressor import NNRegressor
from src.gb_classifier import GBClassifier

warnings.simplefilter(action='ignore', category=FutureWarning)

### CONFIGURATION

In [14]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]
target = active_dataset_config["target"]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

raw_train, raw_test = data_manager.load_raw_data()
raw_train = processor.cut_data(raw_train)

### STRATIFIED CROSS-VALIDATION

In [3]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def my_cross_validation(model, data, target, skf, processor):
    log_loss_scores = []
    accuracy_scores = []
    oof_preds = np.zeros(len(data))

    for train_idx, valid_idx in skf.split(data, data[target]):
        train_df = raw_train.iloc[train_idx]
        valid_df = raw_train.iloc[valid_idx]

        train_proc = processor.fit_transform(train_df)
        valid_proc = processor.transform(valid_df)

        X_train, y_train = train_proc.drop(columns=[target]), train_proc[target]
        X_valid, y_valid = valid_proc.drop(columns=[target]), valid_proc[target]

        model.fit(X_train, y_train)

        preds = model.predict(X_valid)
        oof_preds[valid_idx] = preds

        fold_log_loss = log_loss(y_valid, preds)
        fold_accuracy = accuracy_score(y_valid, preds)
        
        log_loss_scores.append(fold_log_loss)
        accuracy_scores.append(fold_accuracy)
    
    return log_loss_scores, accuracy_scores, oof_preds

### LOGISTIC REGRESSION

In [4]:
%%time

logistic = LogisticRegression()
logistic_log_loss_scores, logistic_accuracy_scores, logistic_oof_preds = my_cross_validation(logistic, raw_train, target, skf, processor)

CPU times: total: 3.06 s
Wall time: 2.15 s


In [5]:
print("Logistic regression log loss:", logistic_log_loss_scores)
print("Logistic regression accuracy:", logistic_accuracy_scores)

Logistic regression log loss: [4.015262987547651, 3.8981211140330196, 4.110778669028812, 4.063921919622959, 4.247744551907457]
Logistic regression accuracy: [0.8886, 0.89185, 0.88595, 0.88725, 0.88215]


### DECISION TREE

In [15]:
%%time

dt = DecisionTreeClassifier(max_depth=8, random_state=42)
dt_log_loss_scores, dt_accuracy_scores, dt_oof_preds = my_cross_validation(dt, raw_train, target, skf, processor)

CPU times: total: 2.38 s
Wall time: 2.37 s


In [16]:
print("Decision tree log loss:", dt_log_loss_scores)
print("Decision tree accuracy:", dt_accuracy_scores)

Decision tree log loss: [4.483830481606174, 4.327040589363515, 4.471215202919983, 4.3702929734304545, 4.564928701731688]
Decision tree accuracy: [0.8756, 0.87995, 0.87595, 0.87875, 0.87335]


### RANDOM FOREST

In [19]:
%%time

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_log_loss_scores, rf_accuracy_scores, rf_oof_preds = my_cross_validation(rf, raw_train, target, skf, processor)

CPU times: total: 17.3 s
Wall time: 17.4 s


In [20]:
print("Random forest log loss:", rf_log_loss_scores)
print("Random forest accuracy:", rf_accuracy_scores)

Random forest log loss: [4.226118359873986, 4.080141563648062, 4.26756856127147, 4.26396419593256, 4.4766217509283495]
Random forest accuracy: [0.88275, 0.8868, 0.8816, 0.8817, 0.8758]


### NEURAL NETWORK

In [6]:
class MLP(NNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_valid: np.ndarray,
        y_valid: np.ndarray, 
        patience: int = 10
    ) -> None:
        input_size = X_train.shape[1]
        
        unique_classes = np.unique(y_train)
        output_size = len(unique_classes)
        
        self._get_network(input_size, output_size)
        
        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).to(torch.long).to(self.device).squeeze()

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), lr=self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            train_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                loss = criterion(preds, batch_y.to(torch.long).to(self.device).squeeze())
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

                pred_labels = val_preds.argmax(dim=1)
                val_acc = (pred_labels == y_valid_t).float().mean().item()

            if (epoch + 1) % 10 == 0:
                print(f"Epoch: {epoch + 1} | Validation log loss: {val_loss:.4f} | Validation accuracy {val_acc:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        X_t = torch.from_numpy(X).to(torch.float32).to(self.device)
        self.eval()
        with torch.no_grad():
            logits = self.forward(X_t)
            predictions = torch.argmax(logits, dim=1)
        
        return predictions.cpu().numpy()

In [7]:
%%time

mlp = MLP(epochs=100, learning_rate=0.0001, hidden_size=[16, 8], batch_size=256)

mlp_log_loss_scores = []
mlp_accuracy_scores = []
mlp_oof_preds = np.zeros(len(raw_train))

for train_idx, valid_idx in skf.split(raw_train, raw_train[target]):
    train_df = raw_train.iloc[train_idx]
    valid_df = raw_train.iloc[valid_idx]

    train_proc = processor.fit_transform(train_df)
    valid_proc = processor.transform(valid_df)

    X_train, y_train = train_proc.drop(columns=[target]), train_proc[target]
    X_valid, y_valid = valid_proc.drop(columns=[target]), valid_proc[target]

    mlp.fit(X_train.values, y_train.values, X_valid.values, y_valid.values)

    preds = mlp.predict(X_valid.values).ravel()
    mlp_oof_preds[valid_idx] = preds

    fold_log_loss = log_loss(y_valid, preds)
    fold_accuracy = accuracy_score(y_valid, preds)
        
    mlp_log_loss_scores.append(fold_log_loss)
    mlp_accuracy_scores.append(fold_accuracy)

    print('-' * 50)

Epoch: 10 | Validation log loss: 0.2923 | Validation accuracy 0.8791
Epoch: 20 | Validation log loss: 0.2779 | Validation accuracy 0.8849
Epoch: 30 | Validation log loss: 0.2726 | Validation accuracy 0.8873
Epoch: 40 | Validation log loss: 0.2703 | Validation accuracy 0.8882
Epoch: 50 | Validation log loss: 0.2697 | Validation accuracy 0.8888
Epoch: 60 | Validation log loss: 0.2697 | Validation accuracy 0.8892
Epoch: 70 | Validation log loss: 0.2694 | Validation accuracy 0.8888
Epoch: 80 | Validation log loss: 0.2693 | Validation accuracy 0.8891
Epoch: 90 | Validation log loss: 0.2694 | Validation accuracy 0.8892
Epoch: 100 | Validation log loss: 0.2693 | Validation accuracy 0.8892
--------------------------------------------------
Epoch: 10 | Validation log loss: 0.2824 | Validation accuracy 0.8845
Epoch: 20 | Validation log loss: 0.2712 | Validation accuracy 0.8900
Epoch: 30 | Validation log loss: 0.2682 | Validation accuracy 0.8904
Epoch: 40 | Validation log loss: 0.2674 | Validatio

In [9]:
print("Neural network log loss:", mlp_log_loss_scores)
print("Neural network accuracy:", mlp_accuracy_scores)

Neural network log loss: [3.997241160853092, 3.914340758058123, 4.101767755681532, 4.065724102292415, 4.272975109279838]
Neural network accuracy: [0.8891, 0.8914, 0.8862, 0.8872, 0.88145]


### GRADIENT BOOSTING (DTs)

In [5]:
%%time

gb_log_loss_scores = []
gb_accuracy_scores = []
gb_oof_preds = np.zeros(len(raw_train))

for train_idx, valid_idx in skf.split(raw_train, raw_train[target]):
    train_df = raw_train.iloc[train_idx]
    valid_df = raw_train.iloc[valid_idx]

    train_proc = processor.fit_transform(train_df)
    valid_proc = processor.transform(valid_df)

    X_train, y_train = train_proc.drop(columns=[target]), train_proc[target]
    X_valid, y_valid = valid_proc.drop(columns=[target]), valid_proc[target]

    gb = GBClassifier(
        n_estimators=1000,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        early_stopping_rounds=25,
        weak_learner_key="decision_tree",
        weak_learner_config={"max_depth": 8}
    )
    
    gb.fit(X_train.values, y_train.values, X_valid.values, y_valid.values)

    preds = gb.predict(X_valid.values).ravel()
    gb_oof_preds[valid_idx] = preds

    fold_log_loss = log_loss(y_valid, preds)
    fold_accuracy = accuracy_score(y_valid, preds)
    
    gb_log_loss_scores.append(fold_log_loss)
    gb_accuracy_scores.append(fold_accuracy)
    
    print('-' * 50)

2026-04-23 20:43:41,860 - INFO - Iteration: 10 | Validation Log Loss: 0.510738 | Validation Accuracy: 0.878150
2026-04-23 20:43:43,281 - INFO - Iteration: 20 | Validation Log Loss: 0.423002 | Validation Accuracy: 0.882650
2026-04-23 20:43:44,718 - INFO - Iteration: 30 | Validation Log Loss: 0.374294 | Validation Accuracy: 0.883800
2026-04-23 20:43:46,187 - INFO - Iteration: 40 | Validation Log Loss: 0.345509 | Validation Accuracy: 0.885500
2026-04-23 20:43:47,800 - INFO - Iteration: 50 | Validation Log Loss: 0.326821 | Validation Accuracy: 0.885450
2026-04-23 20:43:49,384 - INFO - Iteration: 60 | Validation Log Loss: 0.314176 | Validation Accuracy: 0.886450
2026-04-23 20:43:50,930 - INFO - Iteration: 70 | Validation Log Loss: 0.305286 | Validation Accuracy: 0.886000
2026-04-23 20:43:52,556 - INFO - Iteration: 80 | Validation Log Loss: 0.298727 | Validation Accuracy: 0.886700
2026-04-23 20:43:54,085 - INFO - Iteration: 90 | Validation Log Loss: 0.294009 | Validation Accuracy: 0.886750
2

--------------------------------------------------


2026-04-23 20:45:02,153 - INFO - Iteration: 10 | Validation Log Loss: 0.507763 | Validation Accuracy: 0.881900
2026-04-23 20:45:03,749 - INFO - Iteration: 20 | Validation Log Loss: 0.419849 | Validation Accuracy: 0.886850
2026-04-23 20:45:05,279 - INFO - Iteration: 30 | Validation Log Loss: 0.371338 | Validation Accuracy: 0.888600
2026-04-23 20:45:06,818 - INFO - Iteration: 40 | Validation Log Loss: 0.342262 | Validation Accuracy: 0.889100
2026-04-23 20:45:08,364 - INFO - Iteration: 50 | Validation Log Loss: 0.323730 | Validation Accuracy: 0.889850
2026-04-23 20:45:09,937 - INFO - Iteration: 60 | Validation Log Loss: 0.311280 | Validation Accuracy: 0.890100
2026-04-23 20:45:11,467 - INFO - Iteration: 70 | Validation Log Loss: 0.302548 | Validation Accuracy: 0.890050
2026-04-23 20:45:12,949 - INFO - Iteration: 80 | Validation Log Loss: 0.296120 | Validation Accuracy: 0.890750
2026-04-23 20:45:14,473 - INFO - Iteration: 90 | Validation Log Loss: 0.291469 | Validation Accuracy: 0.890450
2

--------------------------------------------------


2026-04-23 20:46:04,065 - INFO - Iteration: 10 | Validation Log Loss: 0.511012 | Validation Accuracy: 0.878100
2026-04-23 20:46:05,676 - INFO - Iteration: 20 | Validation Log Loss: 0.422778 | Validation Accuracy: 0.882700
2026-04-23 20:46:07,219 - INFO - Iteration: 30 | Validation Log Loss: 0.375116 | Validation Accuracy: 0.883650
2026-04-23 20:46:08,877 - INFO - Iteration: 40 | Validation Log Loss: 0.346786 | Validation Accuracy: 0.884850
2026-04-23 20:46:10,935 - INFO - Iteration: 50 | Validation Log Loss: 0.328866 | Validation Accuracy: 0.884950
2026-04-23 20:46:12,993 - INFO - Iteration: 60 | Validation Log Loss: 0.316962 | Validation Accuracy: 0.885100
2026-04-23 20:46:14,839 - INFO - Iteration: 70 | Validation Log Loss: 0.308587 | Validation Accuracy: 0.884950
2026-04-23 20:46:16,632 - INFO - Iteration: 80 | Validation Log Loss: 0.302512 | Validation Accuracy: 0.884800
2026-04-23 20:46:18,408 - INFO - Iteration: 90 | Validation Log Loss: 0.298024 | Validation Accuracy: 0.885350
2

--------------------------------------------------


2026-04-23 20:47:06,947 - INFO - Iteration: 10 | Validation Log Loss: 0.508245 | Validation Accuracy: 0.881100
2026-04-23 20:47:08,502 - INFO - Iteration: 20 | Validation Log Loss: 0.420265 | Validation Accuracy: 0.884350
2026-04-23 20:47:10,010 - INFO - Iteration: 30 | Validation Log Loss: 0.372182 | Validation Accuracy: 0.884800
2026-04-23 20:47:11,542 - INFO - Iteration: 40 | Validation Log Loss: 0.343425 | Validation Accuracy: 0.885250
2026-04-23 20:47:13,067 - INFO - Iteration: 50 | Validation Log Loss: 0.325139 | Validation Accuracy: 0.885750
2026-04-23 20:47:14,531 - INFO - Iteration: 60 | Validation Log Loss: 0.312909 | Validation Accuracy: 0.886600
2026-04-23 20:47:16,059 - INFO - Iteration: 70 | Validation Log Loss: 0.304418 | Validation Accuracy: 0.886200
2026-04-23 20:47:17,621 - INFO - Iteration: 80 | Validation Log Loss: 0.298251 | Validation Accuracy: 0.887150
2026-04-23 20:47:19,133 - INFO - Iteration: 90 | Validation Log Loss: 0.293683 | Validation Accuracy: 0.886800
2

--------------------------------------------------


2026-04-23 20:48:10,564 - INFO - Iteration: 10 | Validation Log Loss: 0.509550 | Validation Accuracy: 0.876650
2026-04-23 20:48:12,111 - INFO - Iteration: 20 | Validation Log Loss: 0.423951 | Validation Accuracy: 0.879750
2026-04-23 20:48:13,658 - INFO - Iteration: 30 | Validation Log Loss: 0.377053 | Validation Accuracy: 0.882350
2026-04-23 20:48:15,193 - INFO - Iteration: 40 | Validation Log Loss: 0.348775 | Validation Accuracy: 0.882750
2026-04-23 20:48:16,702 - INFO - Iteration: 50 | Validation Log Loss: 0.330714 | Validation Accuracy: 0.882850
2026-04-23 20:48:18,362 - INFO - Iteration: 60 | Validation Log Loss: 0.318919 | Validation Accuracy: 0.882650
2026-04-23 20:48:19,850 - INFO - Iteration: 70 | Validation Log Loss: 0.310633 | Validation Accuracy: 0.883050
2026-04-23 20:48:21,354 - INFO - Iteration: 80 | Validation Log Loss: 0.304683 | Validation Accuracy: 0.882900
2026-04-23 20:48:22,900 - INFO - Iteration: 90 | Validation Log Loss: 0.300228 | Validation Accuracy: 0.883650
2

--------------------------------------------------
CPU times: total: 5min 21s
Wall time: 5min 26s


In [6]:
print("Gradient boosting (DTs) log loss:", gb_log_loss_scores)
print("Gradient boosting (DTs) accuracy:", gb_accuracy_scores)

Gradient boosting (DTs) log loss: [4.049504458267312, 3.943175680769417, 4.143217957079017, 4.063921919622959, 4.215305263857251]
Gradient boosting (DTs) accuracy: [0.88765, 0.8906, 0.88505, 0.88725, 0.88305]


### XGBOOST

In [16]:
%%time

xgb_log_loss_scores = []
xgb_accuracy_scores = []
xgb_oof_preds = np.zeros(len(raw_train))

for train_idx, valid_idx in skf.split(raw_train, raw_train[target]):
    train_df = raw_train.iloc[train_idx]
    valid_df = raw_train.iloc[valid_idx]

    train_proc = processor.fit_transform(train_df)
    valid_proc = processor.transform(valid_df)

    X_train, y_train = train_proc.drop(columns=[target]), train_proc[target]
    X_valid, y_valid = valid_proc.drop(columns=[target]), valid_proc[target]

    xgb = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        early_stopping_rounds=25
    )
    
    xgb.fit(X_train.values, y_train.values, eval_set=[(X_valid.values, y_valid.values)], verbose=False)

    preds = xgb.predict(X_valid.values).ravel()
    xgb_oof_preds[valid_idx] = preds

    fold_log_loss = log_loss(y_valid, preds)
    fold_accuracy = accuracy_score(y_valid, preds)
    
    xgb_log_loss_scores.append(fold_log_loss)
    xgb_accuracy_scores.append(fold_accuracy)
    
    print('-' * 50)

--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
CPU times: total: 39 s
Wall time: 2.84 s


In [17]:
print("XGBoost log loss:", xgb_log_loss_scores)
print("XGBoost accuracy:", xgb_accuracy_scores)

XGBoost log loss: [3.930560402083225, 3.8981211140330196, 4.098163390342621, 4.026076083564385, 4.190074706484869]
XGBoost accuracy: [0.89095, 0.89185, 0.8863, 0.8883, 0.88375]


### GRADIENT BOOSTING (NNs)

In [ ]:
%%time

nn_gb_log_loss_scores = []
nn_gb_accuracy_scores = []
nn_gb_oof_preds = np.zeros(len(raw_train))

for train_idx, valid_idx in skf.split(raw_train, raw_train[target]):
    train_df = raw_train.iloc[train_idx]
    valid_df = raw_train.iloc[valid_idx]

    train_proc = processor.fit_transform(train_df)
    valid_proc = processor.transform(valid_df)

    X_train, y_train = train_proc.drop(columns=[target]), train_proc[target]
    X_valid, y_valid = valid_proc.drop(columns=[target]), valid_proc[target]

    nn_gb = GBClassifier(
        n_estimators=1000,
        learning_rate=0.5,
        subsample=1.0,
        colsample_bytree=1.0,
        reg_lambda=0.5,
        early_stopping_rounds=10,
        weak_learner_key="neural_network",
        weak_learner_config={
            "epochs": 30,
            "learning_rate": 0.0001,
            "hidden_size": [16, 8],
            "batch_size": 256
        }
    )
    
    nn_gb.fit(X_train.values, y_train.values, X_valid.values, y_valid.values)

    preds = nn_gb.predict(X_valid.values).ravel()
    nn_gb_oof_preds[valid_idx] = preds

    fold_log_loss = log_loss(y_valid, preds)
    fold_accuracy = accuracy_score(y_valid, preds)
    
    nn_gb_log_loss_scores.append(fold_log_loss)
    nn_gb_accuracy_scores.append(fold_accuracy)
    
    print('-' * 50)